# Clase 165 — RL moderno: A3C, PPO, SAC (vista general)

Panorama de los algoritmos actor-critic modernos: **A3C** (async advantage actor-critic),
**PPO** (el default industrial) y **SAC** (off-policy, acciones continuas). Incluye el cálculo
de **GAE** y del **objetivo clipeado de PPO** en numpy (ejecutable) y el uso de Stable-Baselines3.

Requiere: `numpy`; `stable_baselines3` opcional (no se ejecuta si no está instalado).

## 1. Actor-Critic: dos redes

- **Actor** `π_θ(a|s)`: elige acciones.
- **Critic** `V_φ(s)`: estima el valor del estado.
- **Advantage** `A(s,a) = G − V(s)`: cuánto mejor fue la acción que el promedio.

El actor se actualiza en la dirección del advantage; el critic aprende a predecir el return.

In [ ]:
import numpy as np
np.random.seed(42)

# Ejemplo de "critic" como funcion de valor estimada (aqui, un baseline aprendido simulado)
rewards = np.array([1, 1, 1, 1, 10, 0, 0], dtype=float)
values  = np.array([2, 3, 4, 6, 8, 1, 0, 0], dtype=float)   # V(s_0..s_T), largo T+1 (bootstrap)
print("rewards:", rewards)
print("values :", values)

## 2. GAE — Generalized Advantage Estimation (ejecutable)

GAE (Schulman et al., 2016) combina sesgo y varianza con `λ`:
`A_t = Σ (γλ)^k · δ_{t+k}`, donde `δ_t = r_t + γ·V(s_{t+1}) − V(s_t)`. Es el estimador de
advantage usado por PPO.

In [ ]:
def gae(rewards, values, gamma=0.99, lam=0.95):
    T = len(rewards)
    adv = np.zeros(T, dtype=float)
    running = 0.0
    for t in reversed(range(T)):
        delta = rewards[t] + gamma * values[t + 1] - values[t]   # TD error
        running = delta + gamma * lam * running
        adv[t] = running
    returns = adv + values[:T]                                   # target para el critic
    return adv, returns

adv, returns = gae(rewards, values)
print("advantage GAE:", np.round(adv, 3))
print("returns      :", np.round(returns, 3))

## 3. El objetivo clipeado de PPO (ejecutable)

PPO limita cuánto puede cambiar la policy en cada update:
`L = min( ratio·A, clip(ratio, 1−ε, 1+ε)·A )` con `ratio = π_new(a|s) / π_old(a|s)`.
El clip evita updates destructivos → estabilidad sin necesitar trust-region exacto.

In [ ]:
def ppo_objective(logp_old, logp_new, advantage, eps=0.2):
    ratio = np.exp(logp_new - logp_old)                 # pi_new / pi_old
    unclipped = ratio * advantage
    clipped = np.clip(ratio, 1 - eps, 1 + eps) * advantage
    return np.minimum(unclipped, clipped)               # se maximiza su media

logp_old = np.log([0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5])
logp_new = np.log([0.7, 0.6, 0.4, 0.9, 0.5, 0.3, 0.5])
A = (adv - adv.mean()) / (adv.std() + 1e-8)
surrogate = ppo_objective(logp_old, logp_new, A)
print("objetivo PPO por timestep:", np.round(surrogate, 3))
print("perdida (a minimizar): -mean =", round(-float(surrogate.mean()), 4))

## 4. Comparativa de algoritmos

| Algo | Tipo | Acciones | Idea clave | Cuándo |
|---|---|---|---|---|
| A3C/A2C | on-policy | disc/cont | actor-critic + workers paralelos | baseline histórico |
| **PPO** | on-policy | disc/cont | objetivo clipeado | **default industrial**, RLHF |
| SAC | off-policy | continuas | entropy bonus + off-policy | sample-efficient, robótica |
| DQN | off-policy | discretas | Q-network + replay | control desde píxeles |

PPO es el primero a probar; SAC si se necesita más sample-efficiency en control continuo.

## 5. Uso práctico con Stable-Baselines3

SB3 (PyTorch) implementa estos algoritmos con una API uniforme.

In [ ]:
try:
    from stable_baselines3 import PPO
    model = PPO("MlpPolicy", "CartPole-v1", verbose=0, tensorboard_log="./tb/")
    model.learn(total_timesteps=50_000)
    print("PPO entrenado")
except Exception as e:
    print("stable-baselines3 no instalado:", type(e).__name__)
    print("API:")
    print("  from stable_baselines3 import PPO, SAC")
    print("  PPO('MlpPolicy', 'CartPole-v1').learn(total_timesteps=100_000)")
    print("  SAC('MlpPolicy', 'Pendulum-v1').learn(total_timesteps=20_000)  # acciones continuas")

## 6. Actor y critic como dos cabezas (Keras)

En la práctica actor y critic comparten un tronco y se ramifican en dos cabezas: `π(a|s)`
(softmax) y `V(s)` (escalar). Se muestra la API (no se ejecuta si TF no está instalado).

In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras
    inputs = keras.Input(shape=(4,))
    trunk = keras.layers.Dense(64, activation="relu")(inputs)
    policy_head = keras.layers.Dense(2, activation="softmax", name="pi")(trunk)   # actor
    value_head  = keras.layers.Dense(1, name="V")(trunk)                          # critic
    ac = keras.Model(inputs, [policy_head, value_head])
    print("actor-critic:", [o.shape for o in ac.outputs])
except Exception as e:
    print("tensorflow no instalado:", type(e).__name__)
    print("actor: Dense(2, softmax) ; critic: Dense(1) ; tronco compartido")

## Ejercicios

1. Entrenar `PPO('MlpPolicy', 'LunarLander-v3')` 500k timesteps y reportar el reward medio de
   100 episodios (criterio resuelto: ≥ 200).
2. Comparar PPO vs SAC en `Pendulum-v1`: sample efficiency y return final.
3. Variar `λ` de GAE en {0.9, 0.95, 1.0} y observar el trade-off sesgo/varianza del advantage.
4. Variar `ε` del clip de PPO en {0.1, 0.2, 0.3} y discutir el efecto en la estabilidad.

## Conclusiones

- Los métodos modernos son actor-critic: actor (policy) + critic (valor), guiados por el advantage.
- GAE estima el advantage balanceando sesgo y varianza con `γ` y `λ`.
- PPO clipea el ratio `π_new/π_old` para updates estables; es el default y la base de RLHF.
- SAC (off-policy, continuo, con bonus de entropía) es más sample-efficient; Stable-Baselines3
  ofrece todos con una API común.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios del README. Las de **núcleo numérico** son ejecutables (con `assert` de verificación); las de frameworks/servicios no instalados aquí (TF, PyTorch, diffusers, Gymnasium, GCP…) se muestran como **código real de referencia** listo para copiar en un entorno con esas dependencias.

### Ejercicio 0 (núcleo) — Objetivo *clip* de PPO en NumPy (ejecutable)

El corazón de PPO es el **surrogate recortado**: `L = min(r_t·A_t, clip(r_t, 1−ε, 1+ε)·A_t)`, donde `r_t = π_new/π_old`. El recorte impide pasos de policy demasiado grandes. Verificamos sus propiedades.

In [ ]:
import numpy as np
rng = np.random.default_rng(2)

def ppo_clip_objective(ratio, adv, eps=0.2):
    unclipped = ratio * adv
    clipped = np.clip(ratio, 1 - eps, 1 + eps) * adv
    return np.minimum(unclipped, clipped)

adv = rng.standard_normal(8)
ratio = np.exp(rng.standard_normal(8) * 0.3)   # ratios cerca de 1
eps = 0.2
L = ppo_clip_objective(ratio, adv, eps)

# El objetivo nunca premia por encima del no-recortado (es un min)
assert np.all(L <= ratio * adv + 1e-12)
# Con A>0 el beneficio se topa en (1+eps)*A; con A<0 se topa en (1-eps)*A
for a, r, l in zip(adv, ratio, L):
    bound = (1 + eps) * a if a > 0 else (1 - eps) * a
    assert l <= bound + 1e-9
print('L_clip =', np.round(L, 3))
print('OK: el recorte de PPO acota el incentivo a mover la policy')

### Ejercicio 1 — PPO con Stable-Baselines3

```python
from stable_baselines3 import PPO
model = PPO('MlpPolicy', 'CartPole-v1', verbose=1)
model.learn(total_timesteps=50_000)
from stable_baselines3.common.evaluation import evaluate_policy
print(evaluate_policy(model, model.get_env(), n_eval_episodes=20))  # ~500 (max)
```

### Ejercicio 2 — SAC en Pendulum

```python
from stable_baselines3 import SAC
model = SAC('MlpPolicy', 'Pendulum-v1', verbose=1).learn(20_000)
# SAC (off-policy, con entropia) es fuerte en control continuo.
```

### Ejercicio 3 — Comparar PPO vs SAC en LunarLander

```python
# PPO (on-policy): estable, mas muestras. SAC (off-policy): mas sample-efficient.
for Algo in (PPO, SAC):
    m = Algo('MlpPolicy', 'LunarLanderContinuous-v2').learn(100_000)
    print(Algo.__name__, evaluate_policy(m, m.get_env(), n_eval_episodes=10))
```

### Ejercicio 4 — TensorBoard

```python
model = PPO('MlpPolicy', 'CartPole-v1', tensorboard_log='./tb/').learn(50_000)
# tensorboard --logdir ./tb/   -> curvas de reward, loss, entropia
```

### Ejercicio 5 — Custom callback (`EvalCallback`)

```python
from stable_baselines3.common.callbacks import EvalCallback
eval_cb = EvalCallback(eval_env, best_model_save_path='./best/',
                       eval_freq=5_000, n_eval_episodes=10)
model.learn(100_000, callback=eval_cb)   # guarda el mejor modelo
```